# GIOŚ - warstwa BRONZE

Cel: zamiana plików XLSX z warstwy RAW na jedną tabelę w formacie długim (1 wiersz = 1 pomiar) i zapis do Parquet, na którym pracuje DuckDB.

Wynik: pliki `pomiary_2024.parquet`, `stacje.parquet` i `stanowiska.parquet` w folderze `data/bronze/`.

## 1. Ścieżki i otwarcie archiwum

Ustalamy katalog projektu (działa z `notebooks/` i z katalogu głównego) oraz otwieramy ZIP z warstwy RAW.

In [1]:
from pathlib import Path
import zipfile
import polars as pl

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
ZIP_PATH = ROOT / 'data' / 'raw' / '2024' / 'pomiary_2024.zip'
BRONZE_DIR = ROOT / 'data' / 'bronze'

archive = zipfile.ZipFile(ZIP_PATH)

for name in archive.namelist():
    print(name)

2024_As(PM10)_24g.xlsx
2024_BaA(PM10)_24g.xlsx
2024_BaP(PM10)_24g.xlsx
2024_BbF(PM10)_24g.xlsx
2024_BjF(PM10)_24g.xlsx
2024_BkF(PM10)_24g.xlsx
2024_C6H6_1g.xlsx
2024_Cd(PM10)_24g.xlsx
2024_CO_1g.xlsx
2024_DBahA(PM10)_24g.xlsx
2024_Depozycja_1m.xlsx
2024_formaldehyd_1g.xlsx
2024_Hg(TGM)_1g.xlsx
2024_IP(PM10)_24g.xlsx
2024_Jony_PM25_24g.xlsx
2024_Ni(PM10)_24g.xlsx
2024_NO_1g.xlsx
2024_NO2_1g.xlsx
2024_NO2_24g.xlsx
2024_NOx_1g.xlsx
2024_O3_1g.xlsx
2024_Pb(PM10)_24g.xlsx
2024_PM10_1g.xlsx
2024_PM10_24g.xlsx
2024_PM25_1g.xlsx
2024_PM25_24g.xlsx
2024_PrekursoryZielonka_1g.xlsx
2024_SO2_1g.xlsx
2024_SO2_24g.xlsx


## 2. Wczytanie jednego pliku i podział na nagłówek i dane

Pliki GIOŚ mają 6 wierszy nagłówka (Nr, Kod stacji, Wskaźnik, Czas uśredniania, Jednostka, Kod stanowiska), a dalej jeden wiersz na każdą godzinę lub dobę.

Wczytujemy wszystko jako tekst (`infer_schema_length=0`).

In [2]:
name = '2024_PM10_1g.xlsx'

raw = pl.read_excel(
    archive.read(name),
    has_header=False,
    infer_schema_length=0,
)

header = raw.head(6)
data = raw.slice(6)

print('cały plik:', raw.shape)
print(header)
print(data.head(3))

cały plik: (8790, 180)
shape: (6, 180)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ column_0  ┆ column_1  ┆ column_2  ┆ column_3  ┆ … ┆ column_17 ┆ column_17 ┆ column_17 ┆ column_1 │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ 6         ┆ 7         ┆ 8         ┆ 79       │
│ str       ┆ str       ┆ str       ┆ str       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆ str       ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ Nr        ┆ 1         ┆ 2         ┆ 3         ┆ … ┆ 176       ┆ 177       ┆ 178       ┆ 179      │
│ Kod       ┆ DsChojnow ┆ DsCzerStr ┆ DsDziePil ┆ … ┆ ZpKoszArK ┆ ZpSzczAnd ┆ ZpSzczecP ┆ ZpSzczPi │
│ stacji    ┆ MalMOB    ┆ aza       ┆ sud       ┆   ┆ raj       ┆ rze       ┆ rze       ┆ lsud     │
│ Wskaźnik  ┆ PM10      ┆ PM10      ┆ PM10      ┆ … 

## 3. Opis kolumn stanowisk

Każda kolumna `column_1…column_N` to jedno stanowisko pomiarowe. Obracamy nagłówek (`transpose`), żeby każde stanowisko było jednym wierszem z kodem stacji, wskaźnikiem i jednostką.

In [3]:
columns_info = (
    header
    .drop('column_0')
    .transpose(include_header=True, header_name='kolumna')
)
columns_info.columns = [
    'kolumna', 'nr', 'kod_stacji', 'wskaznik',
    'czas_usredniania', 'jednostka', 'kod_stanowiska',
]

print(columns_info.head())

shape: (5, 7)
┌──────────┬─────┬─────────────────┬──────────┬──────────────────┬───────────┬─────────────────────┐
│ kolumna  ┆ nr  ┆ kod_stacji      ┆ wskaznik ┆ czas_usredniania ┆ jednostka ┆ kod_stanowiska      │
│ ---      ┆ --- ┆ ---             ┆ ---      ┆ ---              ┆ ---       ┆ ---                 │
│ str      ┆ str ┆ str             ┆ str      ┆ str              ┆ str       ┆ str                 │
╞══════════╪═════╪═════════════════╪══════════╪══════════════════╪═══════════╪═════════════════════╡
│ column_1 ┆ 1   ┆ DsChojnowMalMOB ┆ PM10     ┆ 1g               ┆ ug/m3     ┆ DsChojnowMalMOB-PM1 │
│          ┆     ┆                 ┆          ┆                  ┆           ┆ 0-1g                │
│ column_2 ┆ 2   ┆ DsCzerStraza    ┆ PM10     ┆ 1g               ┆ ug/m3     ┆ DsCzerStraza-PM10-1 │
│          ┆     ┆                 ┆          ┆                  ┆           ┆ g                   │
│ column_3 ┆ 3   ┆ DsDziePilsud    ┆ PM10     ┆ 1g               ┆ ug/m3     

## 4. Zamiana formatu szerokiego na długi

`unpivot` składa kolumny stanowisk w jedną kolumnę `wartosc` (1 wiersz = 1 pomiar), a `join` dokleja opis stanowiska.

Kontrola: liczba wierszy = liczba godzin x liczba stanowisk (dla PM10 1g: 8784 x 179 = 1 572 336).

In [4]:
long = (
    data
    .rename({'column_0': 'data_czas'})
    .unpivot(index='data_czas', variable_name='kolumna', value_name='wartosc')
    .join(columns_info, on='kolumna', how='left')
)

print(long.shape)
print(long.head())

(1572336, 9)
shape: (5, 9)
┌──────────────┬──────────┬─────────┬─────┬───┬──────────┬──────────────┬───────────┬──────────────┐
│ data_czas    ┆ kolumna  ┆ wartosc ┆ nr  ┆ … ┆ wskaznik ┆ czas_usredni ┆ jednostka ┆ kod_stanowis │
│ ---          ┆ ---      ┆ ---     ┆ --- ┆   ┆ ---      ┆ ania         ┆ ---       ┆ ka           │
│ str          ┆ str      ┆ str     ┆ str ┆   ┆ str      ┆ ---          ┆ str       ┆ ---          │
│              ┆          ┆         ┆     ┆   ┆          ┆ str          ┆           ┆ str          │
╞══════════════╪══════════╪═════════╪═════╪═══╪══════════╪══════════════╪═══════════╪══════════════╡
│ 2024-01-01   ┆ column_1 ┆ null    ┆ 1   ┆ … ┆ PM10     ┆ 1g           ┆ ug/m3     ┆ DsChojnowMal │
│ 01:00:00     ┆          ┆         ┆     ┆   ┆          ┆              ┆           ┆ MOB-PM10-1g  │
│ 2024-01-01   ┆ column_1 ┆ null    ┆ 1   ┆ … ┆ PM10     ┆ 1g           ┆ ug/m3     ┆ DsChojnowMal │
│ 02:00:00     ┆          ┆         ┆     ┆   ┆          ┆      

## 5. Typy kolumn i pochodzenie danych

Wybieramy docelowe kolumny i nadajemy typy: `data_czas` na datę (format `%Y-%m-%d %H:%M:%S`), `wartosc` na liczbę (tekst zamiast liczby zatrzyma kod). Dodajemy `plik_zrodlowy` z nazwą pliku XLSX.

Kolumny pomocnicze (`kolumna`, `nr`) usuwamy, bo służyły tylko do połączenia.

In [5]:
bronze = long.select(
    'kod_stanowiska',
    'kod_stacji',
    'wskaznik',
    'czas_usredniania',
    'jednostka',
    pl.col('data_czas').str.to_datetime('%Y-%m-%d %H:%M:%S'),
    pl.col('wartosc').cast(pl.Float64),
    pl.lit(name).alias('plik_zrodlowy'),
)

print(bronze.schema)
print(bronze.head(3))

Schema({'kod_stanowiska': String, 'kod_stacji': String, 'wskaznik': String, 'czas_usredniania': String, 'jednostka': String, 'data_czas': Datetime(time_unit='us', time_zone=None), 'wartosc': Float64, 'plik_zrodlowy': String})
shape: (3, 8)
┌─────────────┬────────────┬──────────┬────────────┬───────────┬────────────┬─────────┬────────────┐
│ kod_stanowi ┆ kod_stacji ┆ wskaznik ┆ czas_usred ┆ jednostka ┆ data_czas  ┆ wartosc ┆ plik_zrodl │
│ ska         ┆ ---        ┆ ---      ┆ niania     ┆ ---       ┆ ---        ┆ ---     ┆ owy        │
│ ---         ┆ str        ┆ str      ┆ ---        ┆ str       ┆ datetime[μ ┆ f64     ┆ ---        │
│ str         ┆            ┆          ┆ str        ┆           ┆ s]         ┆         ┆ str        │
╞═════════════╪════════════╪══════════╪════════════╪═══════════╪════════════╪═════════╪════════════╡
│ DsChojnowMa ┆ DsChojnowM ┆ PM10     ┆ 1g         ┆ ug/m3     ┆ 2024-01-01 ┆ null    ┆ 2024_PM10_ │
│ lMOB-PM10-1 ┆ alMOB      ┆          ┆            ┆ 

## 6. Konwersja wszystkich plików

Kroki 2-5 są zebrane w funkcji `xlsx_to_bronze` w `scripts/bronze_gios.py`. Uruchamiamy ją dla każdego pliku z archiwum i łączymy wyniki jeden pod drugim.

Pomijamy plik `2024_Depozycja_1m.xlsx`, ponieważ:

1. ma inny układ: przesunięty nagłówek i dwie kolumny dat (`Data od`, `Data do`),
2. godzina zapisana jest jako `24:00`, której nie da się odczytać jako daty,
3. to inny rodzaj pomiaru: opad na powierzchnię (ug/m2/dzień) zamiast stężenia w powietrzu (ug/m3),
4. zawiera tylko 25 wierszy z 3 stacji.

Pominięcie opisujemy w ograniczeniach projektu.

In [6]:
import runpy

bronze_module = runpy.run_path(str(ROOT / 'scripts' / 'bronze_gios.py'))
xlsx_to_bronze = bronze_module['xlsx_to_bronze']

SKIP = {'2024_Depozycja_1m.xlsx'}

parts = []
for name in archive.namelist():
    if name in SKIP:
        print('pomijam:', name)
        continue
    part = xlsx_to_bronze(archive, name)
    print(f'{name:34} {part.height:>10,} wierszy')
    parts.append(part)

bronze = pl.concat(parts)

print('RAZEM:', bronze.shape)
print(bronze.null_count())

2024_As(PM10)_24g.xlsx                 20,862 wierszy
2024_BaA(PM10)_24g.xlsx                 6,954 wierszy
2024_BaP(PM10)_24g.xlsx                63,684 wierszy
2024_BbF(PM10)_24g.xlsx                 6,954 wierszy


2024_BjF(PM10)_24g.xlsx                 6,954 wierszy
2024_BkF(PM10)_24g.xlsx                 6,954 wierszy


2024_C6H6_1g.xlsx                     553,392 wierszy
2024_Cd(PM10)_24g.xlsx                 21,594 wierszy


2024_CO_1g.xlsx                       483,120 wierszy
2024_DBahA(PM10)_24g.xlsx               6,954 wierszy
pomijam: 2024_Depozycja_1m.xlsx
2024_formaldehyd_1g.xlsx                8,784 wierszy
2024_Hg(TGM)_1g.xlsx                   43,920 wierszy
2024_IP(PM10)_24g.xlsx                  6,954 wierszy
2024_Jony_PM25_24g.xlsx                18,300 wierszy
2024_Ni(PM10)_24g.xlsx                 21,594 wierszy


2024_NO_1g.xlsx                     1,177,056 wierszy


...

2024_NOx_1g.xlsx                    1,010,160 wierszy


2024_O3_1g.xlsx                       860,832 wierszy
2024_Pb(PM10)_24g.xlsx                 15,372 wierszy


2024_PM10_1g.xlsx                   1,572,336 wierszy
2024_PM10_24g.xlsx                     58,560 wierszy


2024_PM25_1g.xlsx                     843,264 wierszy
2024_PM25_24g.xlsx                     19,764 wierszy
2024_PrekursoryZielonka_1g.xlsx       254,736 wierszy


2024_SO2_1g.xlsx                      685,152 wierszy
2024_SO2_24g.xlsx                         732 wierszy
RAZEM: (8952726, 8)
shape: (1, 8)
┌─────────────┬────────────┬──────────┬─────────────┬───────────┬───────────┬─────────┬────────────┐
│ kod_stanowi ┆ kod_stacji ┆ wskaznik ┆ czas_usredn ┆ jednostka ┆ data_czas ┆ wartosc ┆ plik_zrodl │
│ ska         ┆ ---        ┆ ---      ┆ iania       ┆ ---       ┆ ---       ┆ ---     ┆ owy        │
│ ---         ┆ u32        ┆ u32      ┆ ---         ┆ u32       ┆ u32       ┆ u32     ┆ ---        │
│ u32         ┆            ┆          ┆ u32         ┆           ┆           ┆         ┆ u32        │
╞═════════════╪════════════╪══════════╪═════════════╪═══════════╪═══════════╪═════════╪════════════╡
│ 0           ┆ 0          ┆ 0        ┆ 0           ┆ 0         ┆ 0         ┆ 470793  ┆ 0          │
└─────────────┴────────────┴──────────┴─────────────┴───────────┴───────────┴─────────┴────────────┘


## 7. Zapis pomiarów do Parquet

Zapisujemy tabelę do `data/bronze/pomiary_2024.parquet`. Parquet przechowuje typy kolumn i zajmuje ok. 19 MB, czyli mniej niż ZIP z GIOŚ (49,5 MB).

In [7]:
BRONZE_DIR.mkdir(parents=True, exist_ok=True)
POMIARY_PATH = BRONZE_DIR / 'pomiary_2024.parquet'

bronze.write_parquet(POMIARY_PATH)

print('Zapisano:', POMIARY_PATH)
print(f'Rozmiar: {POMIARY_PATH.stat().st_size / 1_000_000:.1f} MB')

Zapisano: C:\Users\krysq\Documents\Projekty\gios-bigdata\data\bronze\pomiary_2024.parquet
Rozmiar: 19.2 MB


## 8. Metadane stacji i stanowisk

Arkusze STACJE i STANOWISKA z pliku metadanych zapisujemy jako `stacje.parquet` i `stanowiska.parquet`. Zmieniamy nazwy kolumn i nadajemy typy (daty, współrzędne). Duplikaty i wartości `BRAK DANYCH` zostają bez zmian.

In [8]:
STACJE_XLSX = ROOT / 'data' / 'raw' / 'stacje' / 'metadane_stacji.xlsx'

stacje = bronze_module['stations_to_bronze'](STACJE_XLSX)
stanowiska = bronze_module['positions_to_bronze'](STACJE_XLSX)

stacje.write_parquet(BRONZE_DIR / 'stacje.parquet')
stanowiska.write_parquet(BRONZE_DIR / 'stanowiska.parquet')

print('stacje:', stacje.shape)
print('stanowiska:', stanowiska.shape)
print(stacje.head(3))

stacje: (1148, 14)
stanowiska: (5849, 10)
shape: (3, 14)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ kod_stacj ┆ kod_miedz ┆ nazwa_sta ┆ stary_kod ┆ … ┆ miejscowo ┆ adres     ┆ szerokosc ┆ dlugosc  │
│ i         ┆ ynarodowy ┆ cji       ┆ _stacji   ┆   ┆ sc        ┆ ---       ┆ ---       ┆ ---      │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ str       ┆ f64       ┆ f64      │
│ str       ┆ str       ┆ str       ┆ str       ┆   ┆ str       ┆           ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ DsBialka  ┆ null      ┆ Białka    ┆ null      ┆ … ┆ Białka    ┆ null      ┆ 51.197783 ┆ 16.11739 │
│ DsBielGro ┆ null      ┆ Bielawa - ┆ null      ┆ … ┆ Bielawa   ┆ ul. Grota ┆ 50.68251  ┆ 16.61734 │
│ t         ┆           ┆ ul. Grota ┆           ┆   ┆           ┆ Roweckieg ┆           ┆ 8        │
│           ┆           ┆ Roweckie

## 9. Kontrola spójności

Sprawdzamy, czy stacje z pomiarów są w metadanych, czy kody się nie powtarzają i które daty nie dały się odczytać.

In [9]:
kody_pomiary = set(bronze['kod_stacji'].unique())
kody_metadane = set(stacje['kod_stacji'])

print('Stacje z pomiarami w 2024:', len(kody_pomiary))
print('Bez wpisu w metadanych:', sorted(kody_pomiary - kody_metadane))
print()
print(
    'Zdublowane kody stacji:',
    stacje.filter(pl.col('kod_stacji').is_duplicated())['kod_stacji'].unique().to_list(),
)
print(
    'Zdublowane kody stanowisk:',
    stanowiska.filter(pl.col('kod_stanowiska').is_duplicated())['kod_stanowiska'].unique().to_list(),
)

print()
for sheet, table in [('STACJE', stacje), ('STANOWISKA', stanowiska)]:
    raw_sheet = pl.read_excel(STACJE_XLSX, sheet_name=sheet, infer_schema_length=0)
    for source_col, col in [
        ('Data uruchomienia', 'data_uruchomienia'),
        ('Data zamknięcia', 'data_zamkniecia'),
    ]:
        filled = raw_sheet[source_col].str.strip_chars().str.len_chars().fill_null(0) > 0
        lost = raw_sheet[source_col].filter(filled & table[col].is_null())
        print(f'{sheet} / {source_col}: nieczytelne daty -> NULL: {lost.to_list()}')

Stacje z pomiarami w 2024: 286
Bez wpisu w metadanych: ['OpBrzegPoprzMOB', 'ZpSzczLaczna']

Zdublowane kody stacji: ['OpKrap3Maja']
Zdublowane kody stanowisk: ['PdSuwPulaski-IP(PM10)-24g', 'PdSuwPulaski-BjF(PM10)-24g', 'PdSuwPulaski-BbF(PM10)-24g', 'PdSuwPulaski-BkF(PM10)-24g', 'PdSuwPulaski-DBahA(PM10)-24g', 'PdSuwPulaski-BaA(PM10)-24g']

STACJE / Data uruchomienia: nieczytelne daty -> NULL: []
STACJE / Data zamknięcia: nieczytelne daty -> NULL: []
STANOWISKA / Data uruchomienia: nieczytelne daty -> NULL: []
STANOWISKA / Data zamknięcia: nieczytelne daty -> NULL: ['31.12.20025', '31.12.20025', '31.12.20025', '31.12.20025']
